In [15]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/AI_SocialMedia_Student_Health_Dataset_clean.csv")

In [3]:
df.isnull().any()
# 欠損値はなし。

Student_ID                    False
Age                           False
Gender                        False
Education_Level               False
Daily_Social_Media_Hours      False
Daily_AI_Tool_Usage_Hours     False
Sleep_Hours                   False
Physical_Activity_Hours       False
Mental_Health_Score           False
Physical_Health_Score         False
Social_Isolation_Score        False
Burnout_Level                 False
Academic_Performance_Score    False
Academic_Failure_Risk         False
dtype: bool

In [4]:
df.describe()

,Age,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score,Social_Isolation_Score,Academic_Performance_Score,Academic_Failure_Risk
count,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000
mean,19.040667,4.537500,2.584580,6.544205,1.246332,72.513156,88.019761,4.312027,78.314859,0.060000
std,3.767257,2.398494,1.677988,1.269770,1.038349,9.245482,9.684253,1.162432,11.544651,0.237495
min,13.000000,0.000000,0.000000,2.000000,0.000000,32.560000,48.030000,0.970000,23.990000,0.000000
25%,16.000000,2.830000,1.310000,5.680000,0.310000,66.780000,81.520000,3.470000,70.530000,0.000000
50%,19.000000,4.500000,2.510000,6.550000,1.130000,73.760000,89.470000,4.280000,78.590000,0.000000
75%,22.000000,6.180000,3.740000,7.410000,1.950000,79.580000,97.160000,5.120000,86.700000,0.000000
max,25.000000,14.000000,9.500000,11.150000,5.000000,91.760000,99.980000,8.410000,99.980000,1.000000


In [5]:
df.columns

Index(['Student_ID', 'Age', 'Gender', 'Education_Level',
       'Daily_Social_Media_Hours', 'Daily_AI_Tool_Usage_Hours', 'Sleep_Hours',
       'Physical_Activity_Hours', 'Mental_Health_Score',
       'Physical_Health_Score', 'Social_Isolation_Score', 'Burnout_Level',
       'Academic_Performance_Score', 'Academic_Failure_Risk'],
      dtype='str')

In [6]:
df.drop(labels=['Student_ID','Education_Level'],inplace=True,axis=1)

In [7]:
categorical_cols = df.select_dtypes(include='object').columns.to_list()
# tmp = mlb.fit_transform(categorical_cols)
# tmp
categorical_cols

/var/folders/dr/zc7jg9wn4g3_yn_nkr7z35300000gn/T/ipykernel_71918/3694860792.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include='object').columns.to_list()


['Gender', 'Burnout_Level']

In [9]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
order = [["Low", "Moderate", "High", "Severe"]]
ordinal_encoder = OrdinalEncoder(categories=order)
df['Burnout_Level'] = ordinal_encoder.fit_transform(df[['Burnout_Level']])


ValueError: could not convert string to float: 'Low'

In [10]:
df['Gender'].unique()

<StringArray>
['Male', 'Non-binary', 'Female']
Length: 3, dtype: str

In [11]:
# genderだがLabelEncodingではなくOne-Hotを使用する。LabelEncodingだと数値の大きさによって学習する恐れがあるためである。
onehot_enc = OneHotEncoder(dtype=int)
encoded_data = onehot_enc.fit_transform(df[['Gender']])
onehot_df = pd.DataFrame(data=encoded_data.toarray(),columns=onehot_enc.get_feature_names_out(['Gender']))
onehot_df
df = pd.concat([df, onehot_df], axis=1)
df.drop(labels='Gender',inplace=True,axis=1)

In [12]:
df

,Age,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score,Social_Isolation_Score,Burnout_Level,Academic_Performance_Score,Academic_Failure_Risk,Gender_Female,Gender_Male,Gender_Non-binary
0,19,5.32,1.21,7.30,1.41,74.06,98.90,4.05,0.0,88.80,0,0,1,0
1,16,5.21,3.89,7.81,2.48,76.13,89.94,4.65,1.0,50.60,0,0,0,1
2,25,3.61,2.65,6.34,0.06,65.01,76.75,5.28,0.0,96.67,0,0,1,0
3,23,2.93,1.43,6.12,1.37,74.67,87.38,3.66,0.0,85.65,0,0,1,0
4,20,6.52,1.64,5.23,1.14,67.74,88.04,3.25,1.0,71.84,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,18,6.10,0.72,5.63,0.89,74.08,71.92,3.78,1.0,77.43,0,0,0,1
14996,22,4.55,1.20,7.20,0.00,73.72,85.84,5.06,0.0,75.81,0,1,0,0
14997,13,7.83,1.46,6.22,0.64,70.03,86.62,4.76,1.0,89.84,0,0,1,0
14998,20,6.99,3.00,4.24,1.97,57.38,85.34,2.82,3.0,65.44,1,1,0,0


In [13]:
# ピアソン相関係数可視化。
df.corr(method='pearson')['Mental_Health_Score']

Age                           0.004315
Daily_Social_Media_Hours     -0.399802
Daily_AI_Tool_Usage_Hours    -0.108232
Sleep_Hours                   0.310587
Physical_Activity_Hours       0.245876
Mental_Health_Score           1.000000
Physical_Health_Score         0.243513
Social_Isolation_Score       -0.291725
Burnout_Level                -0.509873
Academic_Performance_Score    0.204037
Academic_Failure_Risk        -0.356007
Gender_Female                 0.002960
Gender_Male                  -0.002149
Gender_Non-binary            -0.002076
Name: Mental_Health_Score, dtype: float64

In [17]:
df.columns
X = df.drop(labels='Mental_Health_Score',axis=1)
y = df['Mental_Health_Score']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)